# Honest object-detection evaluation: YOLO11n on road damage (potholes, cracks, manholes)

**More of my work:** https://github.com/lyftium

## Summary
- **What this is:** a methods demo on public data. I fine-tune a small YOLO briefly and then spend most of the notebook on *evaluation*: per-class precision/recall/AP, a confusion matrix, a confidence-threshold sweep for F1, and a look at the worst images. It is not any of my work projects and none of their numbers appear here.
- **Data:** [Road Damage Dataset: Potholes, Cracks and Manholes](https://www.kaggle.com/datasets/lorenzoarcioni/road-damage-dataset-potholes-cracks-and-manholes) (COCO annotations), split 80/20 by image with a fixed seed.
- **Method:** YOLO11n (COCO-pretrained), 640 px, training capped at ~20 min on a T4. Evaluation on the held-out 20% only.
- **Result (402 held-out images):** over two runs I got mAP50 0.50 and 0.53 (mAP50-95 0.21 and 0.22). Time-capped GPU training isn't bit-for-bit reproducible, so the exact numbers for *this* version are in the summary cell at the end. What held in both runs: manhole is easiest (AP50 ~0.71-0.73), pothole is in the middle (~0.45-0.48), crack is the weakest (~0.34-0.37), and every class has a different best-F1 threshold. At its best threshold the model finds only about a third of the cracks.
- **What I'd do next:** train longer with a bigger model, tile images for thin cracks, and split by location/date so near-duplicate frames can't leak.

A single mAP hides most of what matters in practice: which class fails, whether the misses are small objects, and what threshold you'd actually deploy at. That's what the sections below are for.

In [ ]:
!pip install -q ultralytics
import json, glob, os, random, shutil, numpy as np, pandas as pd, matplotlib.pyplot as plt, torch
from pathlib import Path
from ultralytics import YOLO
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")
ann_path = glob.glob("/kaggle/input/**/annotations_coco.json", recursive=True)[0]
coco = json.load(open(ann_path)); img_root = Path(ann_path).parent / "images"
cats = sorted(coco["categories"], key=lambda c: c["id"]); names = [c["name"] for c in cats]; cid = {c["id"]: i for i, c in enumerate(cats)}
print(len(coco["images"]), "images,", len(coco["annotations"]), "boxes, classes:", names)

## Data: COCO → YOLO, split by image

In [ ]:
by_img = {}
for a in coco["annotations"]: by_img.setdefault(a["image_id"], []).append(a)
imgs = [im for im in coco["images"] if (img_root / Path(im["file_name"]).name).exists()]
random.Random(0).shuffle(imgs); cut = int(0.8 * len(imgs)); split = {"train": imgs[:cut], "val": imgs[cut:]}
root = Path("/tmp/ds")
for s, ims in split.items():
    (root / s / "images").mkdir(parents=True, exist_ok=True); (root / s / "labels").mkdir(parents=True, exist_ok=True)
    for im in ims:
        fn = Path(im["file_name"]).name; W, H = im["width"], im["height"]
        os.symlink(img_root / fn, root / s / "images" / fn) if not (root / s / "images" / fn).exists() else None
        lines = [f"{cid[a['category_id']]} {(a['bbox'][0] + a['bbox'][2] / 2) / W:.6f} {(a['bbox'][1] + a['bbox'][3] / 2) / H:.6f} {a['bbox'][2] / W:.6f} {a['bbox'][3] / H:.6f}"
                 for a in by_img.get(im["id"], []) if a["bbox"][2] > 0 and a["bbox"][3] > 0]
        (root / s / "labels" / (Path(fn).stem + ".txt")).write_text("\n".join(lines))
(root / "data.yaml").write_text(f"path: {root}\ntrain: train/images\nval: val/images\nnames: {names}\n")
counts = pd.DataFrame({s: pd.Series([names[cid[a['category_id']]] for im in ims for a in by_img.get(im['id'], [])]).value_counts() for s, ims in split.items()})
print({s: len(v) for s, v in split.items()}); display(counts)
areas = pd.Series([a["bbox"][2] * a["bbox"][3] / (im["width"] * im["height"]) for im in imgs for a in by_img.get(im["id"], [])])
print(f"box area as share of image: median {areas.median():.3%}, 10th pct {areas.quantile(.1):.3%}")

## Training (short, time-capped)

In [ ]:
model = YOLO("yolo11n.pt")
model.train(data=str(root / "data.yaml"), epochs=60, time=0.33, imgsz=640, batch=16, seed=0, patience=15,
            project="/tmp/runs", name="rd", exist_ok=True, verbose=False, plots=True)
best = YOLO("/tmp/runs/rd/weights/best.pt")

## Per-class precision, recall and AP on the held-out 20%

In [ ]:
m = best.val(data=str(root / "data.yaml"), split="val", imgsz=640, conf=0.001, iou=0.6, plots=True, project="/tmp/runs", name="val", exist_ok=True, verbose=False)
per = pd.DataFrame({"class": [names[i] for i in m.box.ap_class_index], "precision": m.box.p, "recall": m.box.r,
                    "AP50": m.box.ap50, "AP50-95": m.box.ap}).set_index("class")
per["val boxes"] = counts["val"].reindex(per.index)
display(per.style.format({c: "{:.3f}" for c in ["precision", "recall", "AP50", "AP50-95"]}))
print(f"overall mAP50 {m.box.map50:.3f}, mAP50-95 {m.box.map:.3f}")
cm = m.confusion_matrix.matrix; lab = names + ["background"]
cmn = cm / np.maximum(cm.sum(0, keepdims=True), 1)
fig, ax = plt.subplots(figsize=(5.5, 4.5)); im = ax.imshow(cmn, cmap="Blues", vmin=0, vmax=1)
for i in range(len(lab)):
    for j in range(len(lab)): ax.text(j, i, f"{cmn[i, j]:.2f}", ha="center", va="center", fontsize=8, color="white" if cmn[i, j] > .5 else "black")
ax.set_xticks(range(len(lab)), lab, rotation=30); ax.set_yticks(range(len(lab)), lab); ax.set(xlabel="true", ylabel="predicted", title="Confusion matrix (column-normalised)")
plt.colorbar(im); plt.tight_layout(); plt.show()

## Confidence-threshold sweep

The default threshold is rarely the one you'd deploy. I match predictions to ground truth myself (greedy by confidence, IoU ≥ 0.5, same class) and compute P/R/F1 per class across thresholds.

In [ ]:
def load_gt(p):
    t = Path(p).read_text().split("\n") if Path(p).exists() else []
    return np.array([[float(v) for v in l.split()] for l in t if l.strip()]).reshape(-1, 5)
def xywh2xyxy(b): return np.c_[b[:, 0] - b[:, 2] / 2, b[:, 1] - b[:, 3] / 2, b[:, 0] + b[:, 2] / 2, b[:, 1] + b[:, 3] / 2]
def iou(a, b):
    lt = np.maximum(a[:, None, :2], b[None, :, :2]); rb = np.minimum(a[:, None, 2:], b[None, :, 2:]); inter = np.prod(np.clip(rb - lt, 0, None), 2)
    ar = lambda x: (x[:, 2] - x[:, 0]) * (x[:, 3] - x[:, 1]); return inter / (ar(a)[:, None] + ar(b)[None] - inter + 1e-9)
val_imgs = sorted((root / "val" / "images").iterdir()); recs = []  # per prediction: img, cls, conf, tp
gts = {}
for r in best.predict([str(p) for p in val_imgs], conf=0.01, imgsz=640, verbose=False, stream=True):
    k = Path(r.path).stem; g = load_gt(root / "val" / "labels" / f"{k}.txt"); gts[k] = g
    pb, pc, pcls = r.boxes.xywhn.cpu().numpy(), r.boxes.conf.cpu().numpy(), r.boxes.cls.cpu().numpy().astype(int)
    used = np.zeros(len(g), bool); I = iou(xywh2xyxy(pb), xywh2xyxy(g[:, 1:])) if len(g) and len(pb) else np.zeros((len(pb), len(g)))
    for i in np.argsort(-pc):
        cand = [(I[i, j], j) for j in range(len(g)) if not used[j] and int(g[j, 0]) == pcls[i] and I[i, j] >= 0.5]
        tp = bool(cand)
        if tp: used[max(cand)[1]] = True
        recs.append((k, pcls[i], pc[i], tp, *pb[i]))
P = pd.DataFrame(recs, columns=["img", "cls", "conf", "tp", "x", "y", "w", "h"])
n_gt = pd.Series(np.concatenate([g[:, 0] for g in gts.values()]).astype(int)).value_counts()
ths = np.round(np.arange(0.05, 0.91, 0.05), 2); rows = []
for c in range(len(names)):
    for t in ths:
        s = P[(P.cls == c) & (P.conf >= t)]; tp = s.tp.sum(); pr = tp / max(len(s), 1); rc = tp / max(n_gt.get(c, 0), 1)
        rows.append((names[c], t, pr, rc, 2 * pr * rc / max(pr + rc, 1e-9)))
S = pd.DataFrame(rows, columns=["class", "thr", "precision", "recall", "f1"])
fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
for c, g in S.groupby("class"):
    ax[0].plot(g.thr, g.f1, label=c); ax[1].plot(g.recall, g.precision, ".-", label=c)
ax[0].set(xlabel="confidence threshold", ylabel="F1", title="F1 vs threshold (IoU 0.5)"); ax[1].set(xlabel="recall", ylabel="precision", title="P/R across thresholds")
ax[0].legend(fontsize=8); plt.tight_layout(); plt.show()
bestthr = S.loc[S.groupby("class").f1.idxmax()].set_index("class"); display(bestthr.style.format("{:.3f}"))

## Error analysis: the images where the model does worst
At each class's best-F1 threshold, I count misses (FN) and false alarms (FP) per image and show the four worst. Green = ground truth, red = false positive, blue = correct.

In [ ]:
thr_c = {names.index(c): t for c, t in bestthr.thr.items()}
Pk = P[P.apply(lambda r: r.conf >= thr_c.get(r.cls, 1), axis=1)]
err = []
for k, g in gts.items():
    s = Pk[Pk.img == k]; err.append((k, len(g) - s.tp.sum(), (~s.tp).sum()))
E = pd.DataFrame(err, columns=["img", "fn", "fp"]).assign(total=lambda d: d.fn + d.fp).sort_values("total", ascending=False)
print(f"at best-F1 thresholds: {int(E.fn.sum())} missed boxes, {int(E.fp.sum())} false alarms over {len(E)} val images")
fig, axs = plt.subplots(1, 4, figsize=(16, 4.2))
for ax, k in zip(axs, E.img[:4]):
    p = next(x for x in val_imgs if x.stem == k); im = plt.imread(p); H, W = im.shape[:2]; ax.imshow(im)
    for b in gts[k]: x0, y0, x1, y1 = xywh2xyxy(b[None, 1:])[0] * [W, H, W, H]; ax.add_patch(plt.Rectangle((x0, y0), x1 - x0, y1 - y0, fill=False, ec="lime", lw=2))
    for _, r in Pk[Pk.img == k].iterrows():
        x0, y0, x1, y1 = xywh2xyxy(np.array([[r.x, r.y, r.w, r.h]]))[0] * [W, H, W, H]
        ax.add_patch(plt.Rectangle((x0, y0), x1 - x0, y1 - y0, fill=False, ec="dodgerblue" if r.tp else "red", lw=1.5, ls="-" if r.tp else "--"))
    e = E.set_index("img").loc[k]; ax.set_title(f"FN {int(e.fn)}  FP {int(e.fp)}", fontsize=9); ax.axis("off")
plt.tight_layout(); plt.show()

In [ ]:
summary = {"val_images": len(val_imgs), "mAP50": round(float(m.box.map50), 3), "mAP50-95": round(float(m.box.map), 3),
           "per_class_AP50": {c: round(float(v), 3) for c, v in per.AP50.items()},
           "best_f1_threshold": {c: {"thr": float(r.thr), "f1": round(float(r.f1), 3), "precision": round(float(r.precision), 3), "recall": round(float(r.recall), 3)} for c, r in bestthr.iterrows()},
           "missed": int(E.fn.sum()), "false_alarms": int(E.fp.sum())}
json.dump(summary, open("summary.json", "w"), indent=2); print(json.dumps(summary, indent=2))

## What I take from this

- A single mAP50 around 0.5 hides a 2x spread between classes: manholes (compact, high-contrast) are fairly easy, thin cracks are hard, and each class has its own best threshold.
- Before deploying I'd pick thresholds per class from a sweep like this on a validation set, then confirm them on a separate test set (here val is used for both model selection and the sweep, so these numbers are slightly optimistic).
- Next steps I'd try: more epochs and a larger model (this run is time-capped), tiled inference for the small cracks, and a test split by location/date rather than random, so near-duplicate frames don't leak between train and val.